In [4]:
!pip install folium plotly pandas openpyxl geopy


Defaulting to user installation because normal site-packages is not writeable


In [5]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
import plotly.express as px
from geopy.geocoders import Nominatim
import time
import plotly.io as pio
pio.renderers.default = "notebook"

In [7]:
def load_and_process_pincode_data(file_path):
    try:
        # Read the Excel file
        df = pd.read_excel(file_path)
        
        # Ensure column names are correctly recognized
        # Convert column names to string and strip whitespace
        df.columns = [str(col).strip() for col in df.columns]
        
        # Check if the expected pincode column exists
        if 'pincode' not in df.columns:
            # Try to find alternative column names
            pincode_alternatives = ['pincode', 'PINCODE', 'Pincode', 'PIN', 'pin', 'Pin Code', 'PIN CODE']
            
            # Find matching column
            pincode_col = None
            for alt in pincode_alternatives:
                if alt in df.columns:
                    pincode_col = alt
                    break
            
            # Rename column if found
            if pincode_col:
                df.rename(columns={pincode_col: 'pincode'}, inplace=True)
            else:
                # If no pincode column found, use the first column
                df.rename(columns={df.columns[0]: 'pincode'}, inplace=True)
        
        # Ensure pincode is treated as string
        df['pincode'] = df['pincode'].astype(str)
        
        # Count occurrences of each pincode
        pincode_counts = df['pincode'].value_counts().reset_index()
        pincode_counts.columns = ['pincode', 'count']
        
        return pincode_counts
    
    except Exception as e:
        print(f"Error loading file: {e}")
        return None

# Load your Excel file
df = load_and_process_pincode_data('hhhh.xlsx')

# Display the processed data
if df is not None:
    print(df.head())
else:
    print("Failed to load data. Please check your file.")


  pincode  count
0  560008      3
1  560096      1


In [8]:
def geocode_pincodes(df, country='India'):
    """
    Convert pincodes to latitude and longitude using Nominatim
    Args:
        df: DataFrame with a 'pincode' column
        country: Country to use for geocoding (default: India)
    Returns:
        DataFrame with 'latitude' and 'longitude' columns added
    """
    # Initialize geocoder
    geolocator = Nominatim(user_agent="pincode_mapper")
    
    # Create empty columns for latitude and longitude
    df['latitude'] = None
    df['longitude'] = None
    
    # Geocode each pincode
    for idx, row in df.iterrows():
        pincode = row['pincode']
        try:
            # Query with pincode and country
            query = f"{pincode}, {country}"
            location = geolocator.geocode(query)
            
            if location:
                df.at[idx, 'latitude'] = location.latitude
                df.at[idx, 'longitude'] = location.longitude
                print(f"Geocoded {pincode}: {location.latitude}, {location.longitude}")
            else:
                print(f"Could not geocode {pincode}")
            
            # Sleep to avoid hitting API limits
            time.sleep(1)
        except Exception as e:
            print(f"Error geocoding {pincode}: {e}")
            time.sleep(1)
            continue
    
    # Drop rows with missing coordinates
    df_clean = df.dropna(subset=['latitude', 'longitude'])
    print(f"Successfully geocoded {len(df_clean)} out of {len(df)} pincodes")
    
    return df_clean

# Geocode the pincodes
# Note: This may take time depending on the number of pincodes
geocoded_df = geocode_pincodes(df)

# Show the geocoded data
geocoded_df.head()


Geocoded 560008: 12.977304199999999, 77.62653690444957
Geocoded 560096: 13.01374545, 77.53559462740469
Successfully geocoded 2 out of 2 pincodes


,pincode,count,latitude,longitude
0,560008,3,12.977304,77.626537
1,560096,1,13.013745,77.535595


In [9]:
def create_folium_map(df):
    """
    Create a Folium bubble map with the geocoded pincode data
    Args:
        df: DataFrame with 'latitude', 'longitude', 'pincode', and 'count' columns
    Returns:
        Folium map object
    """
    # Calculate the center of the map
    center_lat = df['latitude'].mean()
    center_lon = df['longitude'].mean()
    
    # Create a map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=11)
    
    # Add a marker cluster
    marker_cluster = MarkerCluster().add_to(m)
    
    # Add markers for each pincode
    for idx, row in df.iterrows():
        # Calculate bubble size based on count (scaled)
        bubble_radius = min(50, max(5, row['count'] / df['count'].max() * 30))
        
        # Create a circle marker
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=bubble_radius,
            popup=f"<b>Pincode:</b> {row['pincode']}<br>"
                  f"<b>Count:</b> {row['count']}",
            tooltip=f"Pincode: {row['pincode']}",
            fill=True,
            fill_opacity=0.6,
            color='blue',
            fill_color='blue'
        ).add_to(marker_cluster)
    
    return m

# Create the Folium map
folium_map = create_folium_map(geocoded_df)

# Display the map
folium_map


In [10]:
folium_map.save('pincode_bubble_map_folium.html')
print("Folium map saved as 'pincode_bubble_map_folium.html'")

Folium map saved as 'pincode_bubble_map_folium.html'


In [11]:
def create_plotly_map(df):
    """
    Create a Plotly bubble map with the geocoded pincode data
    Args:
        df: DataFrame with 'latitude', 'longitude', 'pincode', and 'count' columns
    Returns:
        Plotly figure object
    """
    # Create the figure
    fig = px.scatter_mapbox(
        df, 
        lat='latitude', 
        lon='longitude', 
        size='count',  # Bubble size based on count
        color='count',  # Color based on count
        hover_name='pincode',  # Hover text shows pincode
        hover_data={
            'count': True,
            'latitude': False,
            'longitude': False
        },
        text='pincode',  # Show pincode on the bubble
        size_max=25,  # Maximum bubble size
        zoom=10,  # Initial zoom level
        mapbox_style='open-street-map',  # Map style (free option)
        title='Pincode Frequency Map',
        color_continuous_scale=px.colors.sequential.Viridis
    )
    
    # Update layout
    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        height=600
    )
    
    # Update hover template
    fig.update_traces(
        hovertemplate='<b>Pincode: %{hovertext}</b><br>Count: %{customdata[0]}'
    )
    
    return fig

# Create the Plotly map
plotly_map = create_plotly_map(geocoded_df)

# Display the map
plotly_map

# Cell 8: Save the Plotly map as HTML and as a PNG image
# Save as HTML
plotly_map.write_html('pincode_bubble_map_plotly.html')
print("Plotly map saved as 'pincode_bubble_map_plotly.html'")

# Save as PNG (requires kaleido package)
try:
    import plotly.io as pio
    pio.write_image(plotly_map, 'pincode_bubble_map_plotly.png')
    print("Plotly map saved as 'pincode_bubble_map_plotly.png'")
except Exception as e:
    print(f"Could not save as PNG: {e}")
    print("To save as PNG, install the kaleido package: pip install kaleido")


Plotly map saved as 'pincode_bubble_map_plotly.html'
Could not save as PNG: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

To save as PNG, install the kaleido package: pip install kaleido
